In [ ]:
'''
%pip install py7zr
import py7zr
import glob

with py7zr.SevenZipFile(r"C:/M4/train.7z", 'r') as opened_file:
    opened_file.extractall(r"C:/M4/train")

all_img_paths = glob.glob(r"C:/M4/train/*.png")
'''

In [ ]:
%pip install -U transformers torch pillow numpy

In [ ]:
from transformers import CLIPProcessor, CLIPModel
import torch
import numpy as np
from PIL import Image

model_16 = CLIPModel.from_pretrained("openai/clip-vit-base-patch16")
model_32 = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor_16 = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch16")
processor_32 = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

def image_embedding(image_path, model, processor):
    """
    Takes an image file path and returns its embedding as a NumPy array.
    """
    # Load and preprocess the image
    image = Image.open(image_path).convert("RGB")
    input = processor(images=image, return_tensors="pt")

    # Extract image features
    with torch.no_grad():
     embeddings = model.get_image_features(**input)
     embeddings = embeddings if isinstance(embeddings, torch.Tensor) else embeddings.pooler_output #for those output that's not tensor yet
    # Normalize and convert to NumPy
    embeddings = embeddings / embeddings.norm(p=2, dim=-1, keepdim=True)
    return embeddings.squeeze().cpu().numpy()


In [ ]:
def create_matrix(image_paths, model=model_16, processor=processor_16): #get the x-train
  embedding_vectors = []
  for image_path in image_paths:
    embedding_vectors.append(image_embedding(image_path, model, processor))

  embedding_matrix = np.stack(embedding_vectors, axis = 0)
  return embedding_matrix


In [ ]:
%pip install -U pandas

In [ ]:
 #sort using csv
import pandas as pd
import os
df = pd.read_csv(r"C:/M4/trainLabels.csv") #read the csv file in the drive
df = df.sort_values(by=df.columns[1]).reset_index(drop=True) #sort by category
print(df)

df_train = pd.DataFrame()
for label in df['label'].unique():
  df_temp = df[df['label'] == label]
  df_temp_part = df_temp.iloc[:len(df_temp)//2] #take first n images of each category as the initial training set
  df_train = pd.concat([df_train, df_temp_part],ignore_index=True)

img_paths = ('C:/M4/train/' + df_train['id'].astype(str) + '.png').tolist()
img_paths

In [ ]:
"""
df['class'] = None

for idx, label in enumerate(df["label"].unique()): #turn string into categories number
    df.loc[df["label"] == label, "class"] = idx

label_list = df['class'].astype(int)
"""

df_train['class'] = None

for idx, label in enumerate(df_train["label"].unique()): #turn string into categories number (from airplane,automobile,...,truck to 0,1,2,...,9)
    df_train.loc[df_train["label"] == label, "class"] = idx

label_list = df_train['class'].astype(int)

In [ ]:
df_train

In [ ]:
import numpy as np
def one_hot_encoded_labels_matrix(labels_list): # grab category list and output elementary-like matrix (y-train)

    num_labels = len(np.unique(labels_list)) # number of categories
    num_samples = len(labels_list)

    Y_onehot = np.zeros((num_samples, num_labels)) # create a zeros matrix
    for sample_idx, sample_label in enumerate(labels_list):

        assert sample_label < num_labels,  f"{sample_label} should be between 0 and {num_labels-1}"
        Y_onehot[sample_idx,sample_label] = 1

    return Y_onehot

In [ ]:
def one_hot_encoded_labels_matrix_same_class(labels_list, num_labels): # grab category list and output elementary-like matrix (y-train)

    num_samples = len(labels_list)

    Y_onehot = np.zeros((num_samples, num_labels)) # create a zeros matrix
    for sample_idx, sample_label in enumerate(labels_list):

        assert sample_label < num_labels,  f"{sample_label} should be between 0 and {num_labels-1}"
        Y_onehot[sample_idx,sample_label] = 1

    return Y_onehot

In [ ]:
x_train_16 = create_matrix(img_paths, model_16, processor_16)

In [ ]:
x_train_32 = create_matrix(img_paths, model_32, processor_32)

In [ ]:
y_train = one_hot_encoded_labels_matrix(label_list)

In [ ]:
x_train_16_tensor = torch.tensor(x_train_16, dtype = torch.float32) #turn into type that model use
x_train_32_tensor = torch.tensor(x_train_32, dtype = torch.float32)
y_train_tensor = torch.tensor(y_train, dtype = torch.float32)

In [ ]:
torch.save(x_train_16_tensor, 'C:/M4/Matrices/x_train_16_tensor.pt')
torch.save(x_train_32_tensor, 'C:/M4/Matrices/x_train_32_tensor.pt')
torch.save(y_train_tensor, 'C:/M4/Matrices/y_train_tensor.pt')

In [ ]:
x_train_16_tensor = torch.load('C:/M4/Matrices/x_train_16_tensor.pt')
x_train_32_tensor = torch.load('C:/M4/Matrices/x_train_32_tensor.pt')
y_train_tensor = torch.load('C:/M4/Matrices/y_train_tensor.pt')

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

def model_fit_pytorch(x_train, y_train, model,loss_criterion = nn.MSELoss(),epochs=1000, lr=0.01, log_interval=100):

    optimizer = optim.SGD(model.parameters(), lr = lr) # SGD is (Stochastic) Gradient Descent

    loss_history = []
    for epoch in range(epochs):

        model.train() # enter training mode
        optimizer.zero_grad() # reset the gradients of all model parameters before computing new gradients

        y_pred = model(x_train)# predict output based on the model

        # Compute loss
        loss = loss_criterion(y_pred, y_train)
        loss.backward() #derivative with respect to each initial variable
        optimizer.step() #perform changes
        loss_history.append(loss.item()) #record loss

        if log_interval and epoch % log_interval == 0:
            print(f"Epoch [{epoch}/{epochs}], Loss: {loss_history[-1]:.4f}") #.4f -> to the fourth decimal place

    return model, loss_history

In [ ]:
x_train_16_tensor.shape

In [ ]:
y_train_tensor.shape

In [ ]:
#nn.Sequential: wraps layers in order (since one linear layer in this case, not required)
#nn.Linear: linear model, y=xW^t+b (x=each row of x_train_tensor)
formula_16 = nn.Sequential(nn.Linear(x_train_16_tensor.shape[1],y_train_tensor.shape[1]))
formula_32 = nn.Sequential(nn.Linear(x_train_16_tensor.shape[1],y_train_tensor.shape[1]))
trained_model_16,loss_history_16 = model_fit_pytorch(x_train_16_tensor, y_train_tensor, formula_16, loss_criterion = nn.MSELoss(),epochs = 3000, lr = 5.5, log_interval=100) #training iterations


In [ ]:
trained_model_32,loss_history_32 = model_fit_pytorch(x_train_32_tensor, y_train_tensor, formula_32, loss_criterion = nn.MSELoss(),epochs = 3000, lr = 5.5, log_interval=100) #training iterations

In [ ]:
%pip install -U seaborn matplotlib

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

def loss_history_plot(loss_history_16, loss_history_32, epochs = None):

    #16
    fig, ax = plt.subplots(1,2,figsize=(12, 5))
    # Loss curve
    ax[0].plot(np.arange(epochs), loss_history_16, marker = "o", color='green', alpha=0.8)
    ax[0].set_yscale('log')
    ax[0].set_title('Loss Over Epochs (Log Scale) CLIP16')
    ax[0].set_xlabel('Epoch')
    ax[0].set_ylabel('Loss')
    ax[0].grid(True)

    y_last_16 = loss_history_16[-1]
    x_last_16 = epochs
    ax[0].scatter(x_last_16,y_last_16)
    ax[0].text(x_last_16, y_last_16, f"final loss: {y_last_16:.4f}\n, total epochs: {x_last_16}")

    # Loss curve
    ax[1].plot(np.arange(epochs), loss_history_32, marker = "o", color='green', alpha=0.8)
    ax[1].set_yscale('log')
    ax[1].set_title('Loss Over Epochs (Log Scale) CLIP32')
    ax[1].set_xlabel('Epoch')
    ax[1].set_ylabel('Loss')
    ax[1].grid(True)

    y_last_32 = loss_history_32[-1]
    x_last_32 = epochs
    ax[1].scatter(x_last_32,y_last_32)
    ax[1].text(x_last_32, y_last_32, f"final loss: {y_last_32:.4f}\n, total epochs: {x_last_32}")

    plt.tight_layout()
    plt.show()

In [ ]:
loss_history_plot(loss_history_16, loss_history_32, 3000)

In [ ]:
def pytorch_model_multiclass_inference(model,X_tensor): #
    model.eval()

    with torch.no_grad():
        Y_predicted = model(X_tensor)
        predicted_labels = torch.argmax(Y_predicted,dim = 1).detach().numpy() #argmax: find the index of maximum values; dim=1: across each row
        #.detach basically remove gradient tracked, since we use no_grad, it's not necessary
    return Y_predicted, predicted_labels

In [ ]:
df_test = pd.DataFrame()
for label in df['label'].unique():
  df_temp = df[df['label'] == label]
  df_temp_part = df_temp.iloc[len(df_temp)//2:len(df_temp)//10*7]
  df_test = pd.concat([df_test, df_temp_part],ignore_index=True)

img_paths_test = ('C:/M4/train/'+df_test['id'].astype(str)+'.png').tolist()
img_paths_test



In [ ]:
df_test

In [ ]:
x_test_16 = create_matrix(img_paths_test, model_16, processor_16)
x_test_32 = create_matrix(img_paths_test, model_32, processor_32)

In [ ]:
x_test_16_tensor = torch.tensor(x_test_16,dtype = torch.float32)
x_test_32_tensor = torch.tensor(x_test_32,dtype = torch.float32)

In [ ]:
torch.save(x_test_16_tensor, 'C:/M4/Matrices/x_test_16_tensor.pt')
torch.save(x_test_32_tensor, 'C:/M4/Matrices/x_test_32_tensor.pt')

In [ ]:
x_test_16_tensor = torch.load('C:/M4/Matrices/x_test_16_tensor.pt')
x_test_32_tensor = torch.load('C:/M4/Matrices/x_test_32_tensor.pt')

In [ ]:
predicted_matrix_16, predicted_label_list_16 = pytorch_model_multiclass_inference(trained_model_16,x_test_16_tensor)
predicted_matrix_32, predicted_label_list_32 = pytorch_model_multiclass_inference(trained_model_32,x_test_32_tensor)

In [ ]:
torch.save(predicted_matrix_16, 'C:/M4/Matrices/predicted_matrix_16.pt')
torch.save(predicted_matrix_32, 'C:/M4/Matrices/predicted_matrix_32.pt')

predicted_matrix_16 = torch.load('C:/M4/Matrices/predicted_matrix_16.pt')
predicted_matrix_32 = torch.load('C:/M4/Matrices/predicted_matrix_32.pt')

In [ ]:
df_test['class'] = None

for idx, label in enumerate(df_test["label"].unique()): #turn string into categories number
    df_test.loc[df_test["label"] == label, "class"] = idx

real_label_list = df_test['class'].astype(int)

In [ ]:
df_hold = pd.DataFrame()
for label in df['label'].unique():
  df_temp = df[df['label'] == label]
  df_temp_part = df_temp.iloc[len(df_temp)//10*7:]
  df_hold = pd.concat([df_hold, df_temp_part],ignore_index=True)

img_paths_hold = ('C:/M4/train/'+df_hold['id'].astype(str)+'.png').tolist()
img_paths_hold

In [ ]:
x_hold_16 = create_matrix(img_paths_hold, model_16, processor_16)
x_hold_32 = create_matrix(img_paths_hold, model_32, processor_32)

In [ ]:
x_hold_16_tensor = torch.tensor(x_hold_16,dtype = torch.float32)
x_hold_32_tensor = torch.tensor(x_hold_32,dtype = torch.float32)

In [ ]:
torch.save(x_hold_16_tensor, 'C:/M4/Matrices/x_hold_16_tensor.pt')
torch.save(x_hold_32_tensor, 'C:/M4/Matrices/x_hold_32_tensor.pt')

In [ ]:
x_hold_16_tensor = torch.load('C:/M4/Matrices/x_hold_16_tensor.pt')
x_hold_32_tensor = torch.load('C:/M4/Matrices/x_hold_32_tensor.pt')

In [ ]:
cat_idx = np.where(df_hold['label'].to_numpy() == 3)[0]
deer_idx = np.where(df_hold['label'].to_numpy() == 4)[0]

In [ ]:
df_cat = df_hold[(df_hold['label'].to_numpy() == 3)]
df_deer = df_hold[(df_hold['label'].to_numpy() == 4)]

In [ ]:
df_cat['class'] = None
df_deer['class'] = None
df_cat['class'] = '3'
df_deer['class'] = '4'

In [ ]:
label_list_cat = df_cat['class'].astype(int)
label_list_deer = df_deer['class'].astype(int)

In [ ]:
df_hold['class'] = None

for idx, label in enumerate(df_hold["label"].unique()): #turn string into categories number
    df_hold.loc[df_hold["label"] == label, "class"] = idx

label_list2 = df_hold['class'].astype(int)

In [ ]:
x_hold_out_cat = x_hold_16_tensor [cat_idx]
x_hold_out_deer = x_hold_32_tensor [deer_idx]

In [ ]:
torch.save(x_hold_out_cat, 'C:/M4/Matrices/x_hold_out_cat.pt')
torch.save(x_hold_out_deer, 'C:/M4/Matrices/x_hold_out_deer.pt')

In [ ]:
x_hold_out_cat = torch.load('C:/M4/Matrices/x_hold_out_cat.pt')
x_hold_out_deer = torch.load('C:/M4/Matrices/x_hold_out_deer.pt')

In [ ]:
y_hold_out_cat = one_hot_encoded_labels_matrix_same_class(label_list_cat,10)
y_hold_out_deer =one_hot_encoded_labels_matrix_same_class(label_list_deer,10)

In [ ]:
y_hold_out_cat_tensor = torch.tensor(y_hold_out_cat,dtype = torch.float32)
y_hold_out_deer_tensor = torch.tensor(y_hold_out_deer,dtype = torch.float32)

In [ ]:
df_hold['class'] = None

for idx, label in enumerate(df_hold["label"].unique()): #turn string into categories number
    df_hold.loc[df_hold["label"] == label, "class"] = idx

label_list2 = df_hold['class'].astype(int)

In [ ]:
y_hold = one_hot_encoded_labels_matrix(label_list2)

In [ ]:
y_hold_tensor = torch.tensor(y_hold,dtype = torch.float32)

In [ ]:
def accuracy(y_pred_label, y_label):
    return np.mean(y_pred_label == y_label)

In [ ]:
accuracy_5to2_16 = accuracy(predicted_label_list_16,real_label_list)
accuracy_5to2_32 = accuracy(predicted_label_list_32,real_label_list)

In [ ]:
def compute_confusion_matrix(correct_labels, predicted_labels):
    unique_labels = np.unique(correct_labels)
    num_labels = len(unique_labels)

    confusion_matrix = np.zeros((num_labels, num_labels))

    for i, unique_label in enumerate(unique_labels):
        true_idxs = (correct_labels == unique_label) #compare every label in correct_labels to a unique label, output a list of T or F
        #true_idxs gives the positions that have label as unique_label
        num_true = np.sum(true_idxs) #count how many trues

        for j, pred_label in enumerate(unique_labels):
            num_pred_as_label = np.sum(predicted_labels[true_idxs] == pred_label)
            if num_true > 0.0:
                confusion_matrix[i, j] = num_pred_as_label/num_true

    return confusion_matrix, unique_labels

In [ ]:
confusion_matrix_16, categories = compute_confusion_matrix(real_label_list, predicted_label_list_16)
confusion_matrix_32, categories = compute_confusion_matrix(real_label_list, predicted_label_list_32)

In [ ]:


def plot_confusion_matrix_heatmap(conf_matrix, class_labels=None, title="Confusion Matrix"):
    plt.figure(figsize=(8, 6))
    sns.heatmap(conf_matrix, annot=True, fmt=".2f", cmap="cividis",  # colorblind friendly palette
                xticklabels=class_labels, yticklabels=class_labels)
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.title(title)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_confusion_matrix_heatmap(confusion_matrix_16, class_labels=df['label'].unique().astype(str), title="Confusion Matrix (clip-vit-base-patch16) 5:2")

In [ ]:
plot_confusion_matrix_heatmap(confusion_matrix_32, class_labels=df['label'].unique().astype(str), title="Confusion Matrix (clip-vit-base-patch32) 5:2")

In [ ]:
x_train_final_16_tensor = torch.cat((x_train_16_tensor, x_hold_16_tensor), dim=0)
x_train_final_32_tensor = torch.cat((x_train_32_tensor, x_hold_32_tensor), dim=0)
y_train_final_tensor = torch.cat((y_train_tensor, y_hold_tensor), dim=0)

In [ ]:
x_train_16_tensor_with_cat = torch.cat((x_train_16_tensor, x_hold_out_cat), dim=0)
x_train_32_tensor_with_deer = torch.cat((x_train_32_tensor, x_hold_out_deer), dim=0)

In [ ]:
y_train_tensor_with_cat = torch.cat((y_train_tensor, y_hold_out_cat_tensor), dim=0)
y_train_tensor_with_deer = torch.cat((y_train_tensor, y_hold_out_deer_tensor), dim=0)

In [ ]:
formula_16_with_cat = nn.Sequential(nn.Linear(x_train_16_tensor.shape[1],y_train_tensor.shape[1]))
formula_32_with_deer = nn.Sequential(nn.Linear(x_train_16_tensor.shape[1],y_train_tensor.shape[1]))
trained_model_16_with_cat,loss_history_16_with_cat = model_fit_pytorch(x_train_16_tensor_with_cat, y_train_tensor_with_cat, formula_16_with_cat, loss_criterion = nn.MSELoss(),epochs = 3000, lr = 5.5, log_interval=100) #training iterations

In [ ]:
trained_model_32_with_deer,loss_history_32_with_deer = model_fit_pytorch(x_train_32_tensor_with_deer, y_train_tensor_with_deer, formula_32_with_deer, loss_criterion = nn.MSELoss(),epochs = 3000, lr = 5.5, log_interval=100) #training iterations

In [ ]:
predicted_matrix_16_with_cat, predicted_label_list_16_with_cat = pytorch_model_multiclass_inference(trained_model_16_with_cat,x_test_16_tensor)
predicted_matrix_32_with_deer, predicted_label_list_32_with_deer = pytorch_model_multiclass_inference(trained_model_32_with_deer,x_test_32_tensor)

In [ ]:
torch.save(predicted_matrix_16_with_cat, 'C:/M4/Matrices/predicted_matrix_16_with_cat.pt')
torch.save(predicted_matrix_32_with_deer, 'C:/M4/Matrices/predicted_matrix_32_with_deer.pt')

predicted_matrix_16_final = torch.load('C:/M4/Matrices/predicted_matrix_16_with_cat.pt')
predicted_matrix_32_final = torch.load('C:/M4/Matrices/predicted_matrix_32_with_deer.pt')

In [ ]:
confusion_matrix_16_with_cat, categories = compute_confusion_matrix(real_label_list, predicted_label_list_16_with_cat)
confusion_matrix_32_with_deer, categories = compute_confusion_matrix(real_label_list, predicted_label_list_32_with_deer)

In [ ]:
plot_confusion_matrix_heatmap(confusion_matrix_16_with_cat, class_labels=df['label'].unique().astype(str), title="Confusion Matrix CLIP16 with extra cat images")

In [ ]:
plot_confusion_matrix_heatmap(confusion_matrix_32_with_deer, class_labels=df['label'].unique().astype(str), title="Confusion Matrix CLIP32 with extra deer images")

In [ ]:
formula_16_final = nn.Sequential(nn.Linear(x_train_16_tensor.shape[1],y_train_tensor.shape[1]))
formula_32_final = nn.Sequential(nn.Linear(x_train_16_tensor.shape[1],y_train_tensor.shape[1]))
trained_model_16_final,loss_history_16_final = model_fit_pytorch(x_train_final_16_tensor, y_train_final_tensor, formula_16_final, loss_criterion = nn.MSELoss(),epochs = 3000, lr = 5.5, log_interval=100) #training iterations


In [ ]:
trained_model_32_final,loss_history_32_final = model_fit_pytorch(x_train_final_32_tensor, y_train_final_tensor, formula_32_final, loss_criterion = nn.MSELoss(),epochs = 3000, lr = 5.5, log_interval=100) #training iterations

In [ ]:
predicted_matrix_16_final, predicted_label_list_16_final = pytorch_model_multiclass_inference(trained_model_16_final,x_test_16_tensor)
predicted_matrix_32_final, predicted_label_list_32_final = pytorch_model_multiclass_inference(trained_model_32_final,x_test_32_tensor)

In [ ]:
torch.save(predicted_matrix_16_final, 'C:/M4/Matrices/predicted_matrix_16_final.pt')
torch.save(predicted_matrix_32_final, 'C:/M4/Matrices/predicted_matrix_32_final.pt')

predicted_matrix_16_final = torch.load('C:/M4/Matrices/predicted_matrix_16_final.pt')
predicted_matrix_32_final = torch.load('C:/M4/Matrices/predicted_matrix_32_final.pt')

In [ ]:
accuracy_4to1_16 = accuracy(predicted_label_list_16_final,real_label_list)
accuracy_4to1_32 = accuracy(predicted_label_list_32_final,real_label_list)

In [ ]:
confusion_matrix_16_final, categories = compute_confusion_matrix(real_label_list, predicted_label_list_16_final)
confusion_matrix_32_final, categories = compute_confusion_matrix(real_label_list, predicted_label_list_32_final)

In [ ]:
plot_confusion_matrix_heatmap(confusion_matrix_16_final, class_labels=df['label'].unique().astype(str), title="Confusion Matrix (clip-vit-base-patch16) 4:1")

In [ ]:
plot_confusion_matrix_heatmap(confusion_matrix_32_final, class_labels=df['label'].unique().astype(str), title="Confusion Matrix (clip-vit-base-patch32) 4:1")

In [ ]:
accuracy_5to2_16 #25000 train w/ 10000 test model 16

In [ ]:
accuracy_5to2_32 #25000 train w/ 10000 test model 32

In [ ]:
accuracy_4to1_16 #40000 train w/ 10000 test model 16

In [ ]:
accuracy_4to1_32 #40000 train w/ 10000 test model 32

In [ ]:
loss_history_plot(loss_history_16_final, loss_history_32_final, 3000)

In [ ]:
%pip install -U random


In [ ]:
df_test['class'] = None

for idx, label in enumerate(df_test["label"].unique()): #turn string into categories number (from airplane,automobile,...,truck to 0,1,2,...,9)
    df_test.loc[df_test["label"] == label, "class"] = idx

In [ ]:
import random
def show_misclassified_example(df_test, idx_of_correct_class, idx_of_wrong_class, predicted_label_list):
        df_the_class = df_test[(df_test['class']==idx_of_correct_class)]

        predicted_label_list_of_class = []
        for idx in df_test.index[(df_test['class']==idx_of_correct_class)]:
            predicted_label_list_of_class.append(predicted_label_list[idx])

        df_wrong = df_the_class[np.array(predicted_label_list_of_class)==idx_of_wrong_class]  

        example_img_path = ('C:/M4/train/' + df_wrong['id'].astype(str) + '.png').tolist()
        
        random_idx = random.randrange(len(example_img_path))
        random_img_path = example_img_path[random_idx]

        img = Image.open(random_img_path)
        plt.imshow(img)
        plt.title(f"True: {df_the_class['label'].iloc[0]} | Predicted: horse | id: {df_wrong['id'].iloc[random_idx]}")
        plt.axis("off")
        plt.show()
        print 



In [ ]:
show_misclassified_example(df_test, 4, 7, predicted_label_list_16_final)